# WIA3003: Academic Project II
## Performance by Recorded Ethnicity

Research question: Does held-out performance differ across the recorded ethnicity groups within this Malaysian dataset? This is a descriptive subgroup evaluation. Ethnicity is joined after prediction for evaluation only. This analysis does not establish causation or validate transfer to another population. Codex assisted with implementation and verification.

## 1.0 Verify the Workbook and Test Records

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

folder = Path.cwd()
project = folder.parent if folder.name == "notebooks" else folder
baseline = project / "results" / "qcnn_baseline"
details = json.loads((baseline / "run_details.json").read_text(encoding="utf-8"))
workbook = Path.home() / "Downloads" / "Orbital, Nasal and Craniofacial Class Pattern.xls"
if hashlib.sha256(workbook.read_bytes()).hexdigest() != details["dataset_sha256"]:
    raise ValueError("Verify the workbook version before using the saved row positions.")

data = pd.read_excel(workbook, sheet_name="Raw Data", usecols=[3, 4])
data.columns = data.columns.str.strip()
assert list(data.columns) == ["Gender", "Ethnicity"]
data["Ethnicity"] = data["Ethnicity"].astype("string").str.strip().str.upper().replace("", pd.NA).fillna("Not recorded")
data["Label"] = data["Gender"].astype("string").str.strip().map(details["label_mapping"])
assert not data["Label"].isna().any()

train_positions = details["train_row_positions"]
test_positions = details["test_row_positions"]
assert np.array_equal(np.sort(train_positions + test_positions), np.arange(len(data)))
test = data.iloc[test_positions].copy()
support = pd.concat({
    "Training": pd.crosstab(data.iloc[train_positions]["Ethnicity"], data.iloc[train_positions]["Gender"].str.strip()),
    "Test": pd.crosstab(test["Ethnicity"], test["Gender"].str.strip()),
}, axis=1)
display(support)

Training      Test    
Gender           F    M    F   M
Ethnicity                       
CHINESE        143  163   33  41
INDIAN          85  145   15  37
MALAY          135  161   44  40

## 2.0 Compare Subgroup Performance

Use the saved predictions on the same 210 records. For each group, recall means the number correctly classified divided by the number actually in that class. Compare both recalls because the sex mix varies by group. Report the counts alongside the scores. No new fitting, feature selection, threshold changes, or formal hypothesis tests are performed here.

In [2]:
models = {
    "Initial QCNN": "qcnn_baseline",
    "CNN": "cnn_baseline",
    "Logistic regression": "logistic_regression_baseline",
}
rows = []
for model, name in models.items():
    run = project / "results" / name
    meta = json.loads((run / "run_details.json").read_text(encoding="utf-8"))
    for field in ("dataset_sha256", "features", "label_mapping", "train_row_positions", "test_row_positions"):
        assert meta[field] == details[field], (model, field)
    saved = pd.read_csv(run / "test_predictions.csv")
    assert saved["source_row_position"].is_unique
    assert set(saved["source_row_position"]) == set(test_positions)
    saved = saved.set_index("source_row_position").loc[test_positions]
    assert np.array_equal(saved["actual_label"], test["Label"])
    assert saved["predicted_label"].isin([0, 1]).all()
    grouped = test.assign(Prediction=saved["predicted_label"].to_numpy())
    for group, part in grouped.groupby("Ethnicity"):
        correct = part["Label"].eq(part["Prediction"])
        female = part["Label"].eq(0)
        male = part["Label"].eq(1)
        rows.append({
            "Model": model, "Recorded ethnicity": group, "Test records": len(part),
            "Correct": int(correct.sum()), "Accuracy": correct.mean(),
            "Female records": int(female.sum()), "Female correct": int((correct & female).sum()),
            "Female recall": correct[female].mean(),
            "Male records": int(male.sum()), "Male correct": int((correct & male).sum()),
            "Male recall": correct[male].mean(),
            "Macro F1": f1_score(part["Label"], part["Prediction"], labels=[0, 1],
                                  average="macro", zero_division=0) if female.any() and male.any() else np.nan,
        })
results = pd.DataFrame(rows)
assert results.groupby("Model")["Test records"].sum().eq(len(test_positions)).all()
display(results.round(4))

,Model,Recorded ethnicity,Test records,Correct,Accuracy,Female records,Female correct,Female recall,Male records,Male correct,Male recall,Macro F1
0,Initial QCNN,CHINESE,74,58,0.7838,33,24,0.7273,41,34,0.8293,0.7798
1,Initial QCNN,INDIAN,52,32,0.6154,15,9,0.6000,37,23,0.6216,0.5853
2,Initial QCNN,MALAY,84,55,0.6548,44,24,0.5455,40,31,0.7750,0.6523
3,CNN,CHINESE,74,61,0.8243,33,25,0.7576,41,36,0.8780,0.8204
4,CNN,INDIAN,52,42,0.8077,15,7,0.4667,37,35,0.9459,0.7292
5,CNN,MALAY,84,65,0.7738,44,33,0.7500,40,32,0.8000,0.7738
6,Logistic regression,CHINESE,74,62,0.8378,33,27,0.8182,41,35,0.8537,0.8359
7,Logistic regression,INDIAN,52,44,0.8462,15,10,0.6667,37,34,0.9189,0.8045
8,Logistic regression,MALAY,84,63,0.7500,44,31,0.7045,40,32,0.8000,0.7500


## 3.0 Interpretation

- Differences describe performance on this one held-out split. They do not prove that ethnicity causes a difference.
- The groups have different numbers of female and male records, so accuracy can reflect the mix of labels as well as model performance.
- Small denominators make class-specific recall sensitive to individual errors. For example, one Indian female record is 1/15 of that subgroup's female recall.
- Use these results for reporting rather than selecting new model settings from the test data.
- XGBoost's available row-level predictions come from a different split, so they are not included. MLP has no matching saved row-level predictions in this workspace.
- A separate labelled dataset would be required for cross-population evaluation.

## 4.0 Save Results

Save the subgroup table and record counts so the Streamlit app can display them. Claude added this step.

In [3]:
output_folder = project / "results" / "ethnicity_evaluation"
output_folder.mkdir(parents=True, exist_ok=True)
results.to_csv(output_folder / "subgroup_metrics.csv", index=False)
support.to_csv(output_folder / "record_counts.csv")
print("Saved results to:", output_folder)

Saved results to: C:\Users\jwenn\Desktop\UM\FYP\FYP-Project\results\ethnicity_evaluation
